# Fall Detection — Real-Time Video Alerting

**What this does:**
- Loads a pre-trained HuggingFace YOLO fall-detection model (no training required)
- Processes a video frame-by-frame with person tracking
- Confirms a fall only after it appears in several consecutive frames (avoids false positives)
- Alerts: prints to console, saves a cropped screenshot, writes a JSON event log
- Saves a fully annotated output video

**Input:** Le2i dataset (Kaggle) or set `VIDEO_INPUT_PATH` to any video file path.

## Section 1: Install + Imports

In [4]:
!pip install -q ultralytics huggingface_hub opencv-python pillow matplotlib
print('Packages ready')

Packages ready


In [5]:
import json
import os
import random
import shutil
import time
from collections import defaultdict, deque
from datetime import datetime
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from huggingface_hub import hf_hub_download, snapshot_download
from ultralytics import YOLO

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | OpenCV {cv2.__version__} | Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch 2.10.0+cu128 | OpenCV 4.13.0 | Device: cuda
GPU: Tesla T4


## Section 2: Config + Model Load

In [6]:
import gc

# ── Paths ──────────────────────────────────────────────────────────────────
BASE        = Path('/kaggle/working')
RESULTS_DIR = BASE / 'fall_alerts'
MODEL_DIR   = BASE / 'hf_model'
for d in [RESULTS_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Model ──────────────────────────────────────────────────────────────────
YOLO_MODEL_ID          = 'melihuzunoglu/human-fall-detection'
YOLO_CONFIDENCE        = 0.35
YOLO_IOU               = 0.45
YOLO_IMGSZ             = 416 if DEVICE == 'cuda' else 320
YOLO_HALF              = DEVICE == 'cuda'

# ── Fall confirmation ──────────────────────────────────────────────────────
CONFIRM_WINDOW         = 10    # sliding window size per person (frames)
CONFIRM_FRAMES         = 6     # detections needed within that window
COOLDOWN_SECONDS       = 5.0   # min gap between alerts for the same person

# ── Processing ────────────────────────────────────────────────────────────
PROCESS_EVERY_N        = 2     # run inference on 1-in-N frames; write all frames
FLUSH_EVERY_N          = 10   # free GPU/CPU memory every N processed frames

# ── Input ──────────────────────────────────────────────────────────────────
VIDEO_INPUT_PATH       = None  # override with a path string to use a specific file

LE2I_INPUT_CANDIDATES = [
    Path('/kaggle/input/datasets/tuyenldvn/falldataset-imvia/Office'),
    Path('/kaggle/input/datasets/tuyenldvn/falldataset-imvia'),
    Path('/kaggle/input/falldataset-imvia/Office'),
    Path('/kaggle/input/falldataset-imvia'),
]

print('Config loaded')

Config loaded


In [7]:
def load_hf_yolo(repo_id, local_dir):
    local_dir = Path(local_dir)
    try:
        weight_path = Path(
            hf_hub_download(repo_id=repo_id, filename='best.pt',
                            repo_type='model', local_dir=str(local_dir))
        )
    except Exception:
        repo_dir = Path(
            snapshot_download(repo_id=repo_id, repo_type='model',
                              local_dir=str(local_dir))
        )
        candidates = sorted(repo_dir.rglob('best.pt')) + sorted(repo_dir.rglob('*.pt'))
        if not candidates:
            raise FileNotFoundError(f'No .pt weights found in {repo_dir}')
        weight_path = candidates[0]

    yolo = YOLO(str(weight_path))
    names = yolo.names if isinstance(yolo.names, dict) else dict(enumerate(yolo.names))
    fall_ids = [
        int(k) for k, v in names.items()
        if any(t in str(v).lower() for t in ['fall', 'fallen', 'lying'])
    ]
    return yolo, names, fall_ids


print(f'Loading model from {YOLO_MODEL_ID} ...')
yolo_model, CLASS_NAMES, FALL_CLASS_IDS = load_hf_yolo(YOLO_MODEL_ID, MODEL_DIR)
print(f'Classes   : {CLASS_NAMES}')
print(f'Fall IDs  : {FALL_CLASS_IDS}')
print('Model ready')

Loading model from melihuzunoglu/human-fall-detection ...


Classes   : {0: 'fallen', 1: 'sitting', 2: 'standing'}
Fall IDs  : [0]
Model ready


## Section 3: Video Selection

In [8]:
VIDEO_EXTS = {'.avi', '.mp4', '.mov', '.mkv', '.wmv'}


def find_videos(root, max_depth=4, max_results=256):
    root = Path(root)
    root_depth = len(root.parts)
    yielded = 0
    for cur, dirs, files in os.walk(root):
        if len(Path(cur).parts) - root_depth >= max_depth:
            dirs[:] = []
        dirs[:] = sorted(dirs)[:64]
        for f in sorted(files):
            if Path(f).suffix.lower() in VIDEO_EXTS:
                yield Path(cur) / f
                yielded += 1
                if yielded >= max_results:
                    return


def resolve_video():
    if VIDEO_INPUT_PATH and Path(VIDEO_INPUT_PATH).exists():
        return Path(VIDEO_INPUT_PATH)

    all_videos = []
    for candidate in LE2I_INPUT_CANDIDATES:
        if candidate.exists():
            all_videos = list(find_videos(candidate))
            if all_videos:
                break

    if not all_videos:
        raise FileNotFoundError(
            'No videos found. Set VIDEO_INPUT_PATH or mount the Le2i dataset.'
        )

    chosen = random.choice(all_videos)
    print(f'Found {len(all_videos)} videos — picked randomly')
    return chosen


VIDEO_PATH = resolve_video()
print(f'Input video: {VIDEO_PATH}')

cap_probe   = cv2.VideoCapture(str(VIDEO_PATH))
fps_probe   = cap_probe.get(cv2.CAP_PROP_FPS) or 25.0
total_probe = int(cap_probe.get(cv2.CAP_PROP_FRAME_COUNT))
w_probe     = int(cap_probe.get(cv2.CAP_PROP_FRAME_WIDTH))
h_probe     = int(cap_probe.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap_probe.release()
print(f'Resolution: {w_probe}x{h_probe} | FPS: {fps_probe:.1f} | Frames: {total_probe}')

Found 33 videos — picked randomly
Input video: /kaggle/input/datasets/tuyenldvn/falldataset-imvia/Office/Office/video (32).avi
Resolution: 320x240 | FPS: 25.0 | Frames: 544


## Section 4: Fall Detection + Alerting

In [9]:
# ── Alert handler ──────────────────────────────────────────────────────────
# Extend this function to add email, Slack, SMS, etc.

def on_fall_detected(person_id, frame_idx, timestamp, confidence, frame, box):
    """
    Called once per confirmed fall event (after cooldown).
    person_id  : tracker-assigned integer ID
    frame_idx  : frame number in the video
    timestamp  : seconds from video start
    confidence : YOLO detection confidence (0-1)
    frame      : full BGR frame (numpy array)
    box        : (x1, y1, x2, y2) bounding box
    """
    ts_str = f'{int(timestamp // 60):02d}:{timestamp % 60:05.2f}'
    msg = (
        f'[FALL ALERT]  Person {person_id} | '
        f'Frame {frame_idx} | Time {ts_str} | Conf {confidence:.2f}'
    )
    print(msg)

    # Save cropped screenshot of the person
    x1, y1, x2, y2 = [int(v) for v in box]
    H, W = frame.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(W, x2), min(H, y2)
    crop = frame[y1:y2, x1:x2] if (x2 > x1 and y2 > y1) else frame
    shot_path = RESULTS_DIR / f'alert_pid{person_id}_frame{frame_idx:06d}.jpg'
    cv2.imwrite(str(shot_path), crop)

    return {
        'person_id':  int(person_id),
        'frame':      int(frame_idx),
        'timestamp':  round(float(timestamp), 3),
        'confidence': round(float(confidence), 4),
        'screenshot': str(shot_path.relative_to(BASE)),
        'box':        [round(float(v), 1) for v in box],
    }


print('Alert handler ready')

Alert handler ready


In [ ]:
# ── Per-person state ───────────────────────────────────────────────────────
detection_windows = defaultdict(lambda: deque(maxlen=CONFIRM_WINDOW))
last_alert_time   = {}

# ── Output video writer ────────────────────────────────────────────────────
OUT_VIDEO = RESULTS_DIR / 'annotated_output.mp4'
fourcc    = cv2.VideoWriter_fourcc(*'mp4v')
writer    = cv2.VideoWriter(str(OUT_VIDEO), fourcc, fps_probe, (w_probe, h_probe))
if not writer.isOpened():
    # fallback codec
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    OUT_VIDEO = RESULTS_DIR / 'annotated_output.avi'
    writer = cv2.VideoWriter(str(OUT_VIDEO), fourcc, fps_probe, (w_probe, h_probe))

event_log     = []
cap           = cv2.VideoCapture(str(VIDEO_PATH))
frame_idx     = 0
infer_idx     = 0   # counts only frames that went through inference
last_vis      = None
t0            = time.time()

print(f'Processing {VIDEO_PATH.name} ...')
print(f'Confirm rule : {CONFIRM_FRAMES} / {CONFIRM_WINDOW} frames | Cooldown: {COOLDOWN_SECONDS}s')
print(f'Inference    : every {PROCESS_EVERY_N} frame(s) | Flush every {FLUSH_EVERY_N} inferences')
print('-' * 60)

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        timestamp = frame_idx / max(fps_probe, 1.0)

        
        if frame_idx % PROCESS_EVERY_N != 0:
            writer.write(last_vis if last_vis is not None else frame)
            frame_idx += 1
            continue

        # ── Run tracker ───────────────────────────────────────────────────
        try:
            results = yolo_model.track(
                frame,
                conf=YOLO_CONFIDENCE,
                iou=YOLO_IOU,
                imgsz=YOLO_IMGSZ,
                half=YOLO_HALF,
                device=0 if DEVICE == 'cuda' else 'cpu',
                tracker='bytetrack.yaml',
                persist=True,
                verbose=False,
            )
        except Exception:
            results = yolo_model.predict(
                frame,
                conf=YOLO_CONFIDENCE,
                iou=YOLO_IOU,
                imgsz=YOLO_IMGSZ,
                half=YOLO_HALF,
                device=0 if DEVICE == 'cuda' else 'cpu',
                verbose=False,
            )

        r         = results[0]
        vis_frame = frame.copy()

        if r.boxes is not None and len(r.boxes) > 0:
            boxes     = r.boxes.xyxy.cpu().numpy()
            classes   = r.boxes.cls.cpu().numpy().astype(int)
            confs     = r.boxes.conf.cpu().numpy()
            track_ids = (
                r.boxes.id.int().cpu().tolist()
                if r.boxes.id is not None
                else list(range(len(boxes)))
            )

            for box, cls_id, det_conf, pid in zip(boxes, classes, confs, track_ids):
                is_fall_det = (
                    cls_id in FALL_CLASS_IDS if FALL_CLASS_IDS
                    else any(t in CLASS_NAMES.get(cls_id, '').lower()
                             for t in ['fall', 'fallen', 'lying'])
                )

                detection_windows[pid].append(1 if is_fall_det else 0)
                confirmed  = sum(detection_windows[pid]) >= CONFIRM_FRAMES
                in_cooldown = (
                    pid in last_alert_time and
                    (timestamp - last_alert_time[pid]) < COOLDOWN_SECONDS
                )

                if confirmed and not in_cooldown:
                    ev = on_fall_detected(
                        person_id=pid,
                        frame_idx=frame_idx,
                        timestamp=timestamp,
                        confidence=float(det_conf),
                        frame=frame,
                        box=box,
                    )
                    event_log.append(ev)
                    last_alert_time[pid] = timestamp

                x1, y1, x2, y2 = [int(v) for v in box]
                color = (0, 0, 255) if confirmed else (0, 200, 0)
                cv2.rectangle(vis_frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(vis_frame,
                            f"{'FALL!' if confirmed else 'OK'} {det_conf:.2f} ID:{pid}",
                            (x1, max(y1 - 6, 14)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        ts_str = f'{int(timestamp // 60):02d}:{timestamp % 60:04.1f}'
        cv2.putText(vis_frame,
                    f'Frame {frame_idx}/{total_probe} | Alerts: {len(event_log)} | {ts_str}',
                    (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        last_vis = vis_frame
        writer.write(vis_frame)

        # ── Periodic memory flush ─────────────────────────────────────────
        del results, r
        infer_idx += 1
        if infer_idx % FLUSH_EVERY_N == 0:
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            elapsed = time.time() - t0
            print(f'  frame {frame_idx}/{total_probe} | '
                  f'{infer_idx / max(elapsed, 0.01):.1f} inf/s | '
                  f'alerts: {len(event_log)}')

        frame_idx += 1

finally:
    cap.release()
    writer.release()
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

elapsed = time.time() - t0
print('-' * 60)
print(f'Done in {elapsed:.1f}s')
print(f'Total fall alerts : {len(event_log)}')
print(f'Annotated video   : {OUT_VIDEO}')

## Section 5: Results

In [ ]:
# ── Save event log ─────────────────────────────────────────────────────────
log_path = RESULTS_DIR / 'event_log.json'
with open(log_path, 'w') as f:
    json.dump(event_log, f, indent=2)
print(f'Event log saved: {log_path}  ({len(event_log)} events)')

# ── Print summary table ────────────────────────────────────────────────────
print()
if event_log:
    print(f'{"Person":>8}  {"Frame":>7}  {"Time":>8}  {"Conf":>6}  Screenshot')
    print('-' * 80)
    for ev in event_log:
        ts = ev['timestamp']
        ts_str = f"{int(ts // 60):02d}:{ts % 60:05.2f}"
        print(f"{ev['person_id']:8d}  {ev['frame']:7d}  {ts_str:>8}  "
              f"{ev['confidence']:6.3f}  {ev['screenshot']}")
else:
    print('No falls detected — try lowering CONFIRM_FRAMES or YOLO_CONFIDENCE.')

In [ ]:
# ── Display alert screenshots ──────────────────────────────────────────────
screenshots = sorted(RESULTS_DIR.glob('alert_pid*.jpg'))

if not screenshots:
    print('No alert screenshots to display.')
else:
    n = min(len(screenshots), 6)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, path in zip(axes, screenshots[:n]):
        img = cv2.imread(str(path))
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(path.stem, fontsize=9)
        ax.axis('off')
    plt.suptitle('Fall Alert Screenshots', fontsize=13)
    plt.tight_layout()
    plt.show()
    print(f'Showing {n} of {len(screenshots)} alert screenshots')

In [ ]:
# ── Sample frames from the annotated output video ─────────────────────────
cap_out  = cv2.VideoCapture(str(OUT_VIDEO))
n_out    = int(cap_out.get(cv2.CAP_PROP_FRAME_COUNT))
picks    = [0, n_out // 4, n_out // 2, 3 * n_out // 4, max(0, n_out - 1)]
picks    = sorted(set(picks))

fig, axes = plt.subplots(1, len(picks), figsize=(4 * len(picks), 4))
if len(picks) == 1:
    axes = [axes]

for ax, fi in zip(axes, picks):
    cap_out.set(cv2.CAP_PROP_POS_FRAMES, fi)
    ret, frm = cap_out.read()
    if ret:
        ax.imshow(cv2.cvtColor(frm, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Frame {fi}', fontsize=9)
    ax.axis('off')

cap_out.release()
plt.suptitle('Annotated Output Video — Key Frames', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── List all output files ──────────────────────────────────────────────────
print('Output files:')
for p in sorted(RESULTS_DIR.rglob('*')):
    if p.is_file():
        print(f'  {str(p.relative_to(BASE)):<55} {p.stat().st_size // 1024:>6} KB')